# E-commerce Recommendation Engine - Models and Evaluation

**Objective:** Build and compare three recommendation approaches on real customer purchase
history: a popularity baseline, item-based collaborative filtering, and ALS matrix
factorization. Evaluate all three on held-out last-basket purchases using ranking metrics.

**Evaluation protocol:** For every customer with 2 or more orders, the most recent order
(invoice) is held out as the test set and every earlier order is used for training. Each
model is asked to rank the full product catalogue and is scored on whether the held-out
items appear in its top 10 recommendations. This is the standard "leave-last-basket-out"
protocol for implicit-feedback recommenders, and avoids the unrealistic optimism of a
random train/test split, which would let a model see future purchases mixed into its
training history.

**Outputs:** `model_results.csv` - Precision@10, Recall@10 and MAP@10 for all three models.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from implicit.als import AlternatingLeastSquares
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print("Setup complete.")

Setup complete.


## 2. Load Cleaned Data

In [2]:
clean = pd.read_csv("../data/clean_transactions.csv", parse_dates=['InvoiceDate'])
print("Loaded:", clean.shape)
clean.head()

Loaded: (396470, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## 3. Leave-Last-Basket-Out Train/Test Split

In [3]:
cust_invoice_dates = clean.groupby(['CustomerID', 'InvoiceNo'])['InvoiceDate'].min().reset_index()
cust_invoice_dates = cust_invoice_dates.sort_values(['CustomerID', 'InvoiceDate'])
cust_invoice_dates['rank_desc'] = cust_invoice_dates.groupby('CustomerID')['InvoiceDate'].rank(
    method='first', ascending=False)

test_invoices = set(cust_invoice_dates.loc[cust_invoice_dates['rank_desc'] == 1, 'InvoiceNo'])
n_orders_per_cust = cust_invoice_dates.groupby('CustomerID')['InvoiceNo'].nunique()
evaluable_customers = set(n_orders_per_cust[n_orders_per_cust >= 2].index)

train = clean[~((clean['InvoiceNo'].isin(test_invoices)) &
                 (clean['CustomerID'].isin(evaluable_customers)))].copy()
test = clean[(clean['InvoiceNo'].isin(test_invoices)) &
             (clean['CustomerID'].isin(evaluable_customers))].copy()

print(f"Train rows: {len(train):,}, Test rows: {len(test):,}")
print(f"Evaluable customers (2+ orders, held-out last basket): {len(evaluable_customers):,}")

train_items = set(train['StockCode'].unique())
test = test[test['StockCode'].isin(train_items)]
test_users = set(test['CustomerID'].unique()) & evaluable_customers
print(f"Test users with at least one in-catalog held-out item: {len(test_users):,}")

Train rows: 337,081, Test rows: 59,389
Evaluable customers (2+ orders, held-out last basket): 2,829
Test users with at least one in-catalog held-out item: 2,827


Held-out items that never appear in training are excluded from evaluation, since no model
could plausibly recommend a product it has never seen — this is a standard cold-start
handling step, not a way of making the task artificially easy; it affected only a small
number of item instances.

## 4. Build the Training Interaction Matrix

In [4]:
user_ids = sorted(train['CustomerID'].unique())
item_ids = sorted(train['StockCode'].unique())
user_idx = {u: i for i, u in enumerate(user_ids)}
item_idx = {it: i for i, it in enumerate(item_ids)}
inv_item_idx = {v: k for k, v in item_idx.items()}
desc_map = clean.groupby('StockCode')['Description'].first()

train_agg = train.groupby(['CustomerID', 'StockCode'])['Quantity'].sum().reset_index()
rows = train_agg['CustomerID'].map(user_idx).values
cols = train_agg['StockCode'].map(item_idx).values
vals = train_agg['Quantity'].values.astype(np.float32)

n_users, n_items = len(user_ids), len(item_ids)
user_item = csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))
print(f"Train interaction matrix: {n_users:,} users x {n_items:,} items, nnz={user_item.nnz:,}")

test_truth = test.groupby('CustomerID')['StockCode'].apply(lambda s: set(s.unique())).to_dict()

Train interaction matrix: 4,334 users x 3,645 items, nnz=228,147


## 5. Model 1 - Popularity Baseline

Recommends the same top-N most-purchased products (by distinct buyer count) to every
customer, excluding items they have already bought. This is the standard non-personalized
baseline: any real personalization technique needs to clear this bar to justify its added
complexity.

In [5]:
item_popularity = np.asarray(user_item.sum(axis=0)).flatten()
top_pop_idx = np.argsort(-item_popularity)

def popularity_recommend(user_row_items, k):
    recs = []
    for idx in top_pop_idx:
        if idx not in user_row_items:
            recs.append(idx)
        if len(recs) == k:
            break
    return recs

print("Popularity model ready.")

Popularity model ready.


## 6. Model 2 - Item-Based Collaborative Filtering

Computes cosine similarity between products based on how many customers bought both
(a binarized, normalized co-purchase matrix), then recommends products most similar to
what a customer has already bought — the classic "customers who bought this also bought"
approach.

In [6]:
binary_ui = (user_item > 0).astype(np.float32)
item_user = binary_ui.T.tocsr()
item_norm = normalize(item_user, axis=1)
item_sim = item_norm.dot(item_norm.T).tocsr()

def itemcf_recommend(user_row, k):
    purchased_idx = user_row.indices
    if len(purchased_idx) == 0:
        return popularity_recommend(set(), k)
    scores = np.asarray(item_sim[purchased_idx].sum(axis=0)).flatten()
    scores[purchased_idx] = -1
    top_idx = np.argpartition(-scores, k)[:k]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    return list(top_idx)

print("Item-based CF model ready. Item similarity matrix:", item_sim.shape)

Item-based CF model ready. Item similarity matrix: (3645, 3645)


## 7. Model 3 - ALS Matrix Factorization

Learns 50 latent factors per user and per item via Alternating Least Squares on
implicit-feedback confidence weights (`confidence = 1 + 2*log(1 + quantity)`), the
standard Hu, Koren and Volinsky (2008) formulation for implicit feedback. Unlike item-based
CF, this can capture latent taste dimensions that are not visible from direct co-purchase
counts alone.

In [7]:
confidence = user_item.copy()
confidence.data = 1.0 + 2.0 * np.log1p(confidence.data)

als_model = AlternatingLeastSquares(factors=50, regularization=0.05, iterations=20, random_state=42)
als_model.fit(confidence)
print("ALS model trained: 50 factors, 20 iterations.")

  0%|          | 0/20 [00:00<?, ?it/s]

ALS model trained: 50 factors, 20 iterations.


## 8. Evaluation - Precision@10, Recall@10, MAP@10

For each test customer, every model ranks the catalogue and is scored on whether its
top-10 recommendations contain the products from that customer's held-out final basket.

In [8]:
K = 10

def evaluate(rec_fn, name, use_user_row=False):
    precisions, recalls, ap_scores = [], [], []
    for u in test_users:
        if u not in user_idx:
            continue
        uidx = user_idx[u]
        truth = {item_idx[it] for it in test_truth[u] if it in item_idx}
        if not truth:
            continue
        purchased_set = set(user_item[uidx].indices)
        recs = rec_fn(user_item[uidx], K) if use_user_row else rec_fn(purchased_set, K)
        hits = [1 if r in truth else 0 for r in recs]
        n_hit = sum(hits)
        precisions.append(n_hit / K)
        recalls.append(n_hit / len(truth))
        if n_hit > 0:
            ap = sum(h * (sum(hits[:i+1]) / (i+1)) for i, h in enumerate(hits)) / min(len(truth), K)
        else:
            ap = 0.0
        ap_scores.append(ap)
    print(f"{name}: n_users={len(precisions)}  Precision@{K}={np.mean(precisions):.4f}  "
          f"Recall@{K}={np.mean(recalls):.4f}  MAP@{K}={np.mean(ap_scores):.4f}")
    return np.mean(precisions), np.mean(recalls), np.mean(ap_scores)

print(f"Evaluating on {len(test_users):,} held-out last-baskets, K={K}\n")
pop_p, pop_r, pop_map = evaluate(popularity_recommend, "Popularity baseline")
cf_p, cf_r, cf_map = evaluate(itemcf_recommend, "Item-based CF", use_user_row=True)

Evaluating on 2,827 held-out last-baskets, K=10



Popularity baseline: n_users=2827  Precision@10=0.0168  Recall@10=0.0089  MAP@10=0.0056


Item-based CF: n_users=2827  Precision@10=0.0467  Recall@10=0.0276  MAP@10=0.0240


In [9]:
als_precisions, als_recalls, als_ap = [], [], []
user_items_csr = user_item.tocsr()

for u in test_users:
    if u not in user_idx:
        continue
    uidx = user_idx[u]
    truth = {item_idx[it] for it in test_truth[u] if it in item_idx}
    if not truth:
        continue
    rec_ids, scores = als_model.recommend(uidx, user_items_csr[uidx], N=K, filter_already_liked_items=True)
    hits = [1 if r in truth else 0 for r in rec_ids]
    n_hit = sum(hits)
    als_precisions.append(n_hit / K)
    als_recalls.append(n_hit / len(truth))
    if n_hit > 0:
        ap = sum(h * (sum(hits[:i+1]) / (i+1)) for i, h in enumerate(hits)) / min(len(truth), K)
    else:
        ap = 0.0
    als_ap.append(ap)

als_p, als_r, als_map = np.mean(als_precisions), np.mean(als_recalls), np.mean(als_ap)
print(f"ALS matrix factorization: n_users={len(als_precisions)}  Precision@{K}={als_p:.4f}  "
      f"Recall@{K}={als_r:.4f}  MAP@{K}={als_map:.4f}")

ALS matrix factorization: n_users=2827  Precision@10=0.0526  Recall@10=0.0322  MAP@10=0.0264


## 9. Results Summary

In [10]:
results = pd.DataFrame([
    {"Model": "Popularity baseline", "Precision@10": pop_p, "Recall@10": pop_r, "MAP@10": pop_map},
    {"Model": "Item-based CF", "Precision@10": cf_p, "Recall@10": cf_r, "MAP@10": cf_map},
    {"Model": "ALS matrix factorization", "Precision@10": als_p, "Recall@10": als_r, "MAP@10": als_map},
])
results.to_csv("model_results.csv", index=False)
results.round(4)

,Model,Precision@10,Recall@10,MAP@10
0,Popularity baseline,0.0168,0.0089,0.0056
1,Item-based CF,0.0467,0.0276,0.0240
2,ALS matrix factorization,0.0526,0.0322,0.0264


In [11]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(results))
width = 0.25
ax.bar(x - width, results['Precision@10'], width, label='Precision@10', color='#4C72B0')
ax.bar(x, results['Recall@10'], width, label='Recall@10', color='#DD8452')
ax.bar(x + width, results['MAP@10'], width, label='MAP@10', color='#55A868')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=10, ha='right')
ax.set_ylabel('Score')
ax.set_title('Recommendation Model Comparison (K=10)')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100)
plt.show()

<Figure size 800x500 with 1 Axes>

Both personalized models clear the popularity baseline by a wide margin, confirming that
purchase-history-based personalization adds real value on top of "just recommend
bestsellers." ALS matrix factorization gives the best ranking quality, since it captures
latent taste patterns that pure co-purchase counting misses, at the cost of being less
directly interpretable than item-based CF.

## 10. Example Recommendations

In [12]:
sample_user = sorted(test_users)[0]
sample_uidx = user_idx[sample_user]
rec_ids, scores = als_model.recommend(sample_uidx, user_items_csr[sample_uidx], N=5,
                                       filter_already_liked_items=True)

print(f"ALS top-5 recommendations for customer {sample_user}:")
for r, s in zip(rec_ids, scores):
    code = inv_item_idx[r]
    print(f"  {code}: {desc_map.get(code, '?').strip():<45} score={s:.4f}")

print(f"\nTheir actual next-basket purchases:")
for code in list(test_truth[sample_user])[:5]:
    print(f"  {code}: {desc_map.get(code, '?').strip()}")

ALS top-5 recommendations for customer 12347:
  84378: SET OF 3 HEART COOKIE CUTTERS                 score=1.1463
  84380: SET OF 3 BUTTERFLY COOKIE CUTTERS             score=1.1296
  21671: RED SPOT CERAMIC DRAWER KNOB                  score=0.9004
  84375: SET OF 20 KIDS COOKIE CUTTERS                 score=0.8955
  84988: SET OF 72 PINK HEART PAPER DOILIES            score=0.8923

Their actual next-basket purchases:
  20719: WOODLAND CHARLOTTE BAG
  21064: BOOM BOX SPEAKER BOYS
  23271: CHRISTMAS TABLE CANDLE SILVER SPIKE
  21265: PINK GOOSE FEATHER TREE 60CM
  23497: CLASSIC CHROME BICYCLE BELL
